# Copy one legacy configuration into the v1 `public` project

Copies a forecast configuration from the legacy warehouse (`iceberg.teehr`, the pre-v1
model) into `public` as v1 datasets, with what it's evaluated against, so the control
plane has real data to evaluate (and we learn how big runs get):

| v1 dataset (`public`) | from (legacy) |
| --- | --- |
| `usgs_gauges` (layer) | `locations`, the gauges the crosswalk links |
| `usgs_streamflow` (observed) | `primary_timeseries`, `OBSERVATIONS`, over the forecast's months |
| `<CONFIGURATION>` (forecast) | `secondary_timeseries`, `CONFIGURATION` |
| `usgs_<prefix>` (crosswalk) | `location_crosswalks` |

**Run it as a platform admin** (the datasets are registered in `public` through the
control plane API, as you). Works on KinD (its legacy warehouse) and use1.

**Spark writes the rows straight into the `public` Iceberg tables**, not through the
API's write path: a configuration is hundreds of millions of rows, and the API takes at
most 64 MB per write. So these writes skip the API's write lineage (the months a write
touched); the first run of an evaluation over them is a full run anyway. A one-time admin
migration, not a pattern (ADR-0004).

**Re-runnable:** registering skips datasets that exist; the layer, crosswalk and series
tables are replaced whole; timeseries are written a month at a time, each replacing that
month's partition (forecasts by month of `reference_time`, observations of
`value_time`). Run again, or from a month on, any time the legacy data changes.

Start with **2. Count**: it writes nothing.

In [ ]:
import os

import requests
import teehr
from pyspark.sql import functions as F
from teehr.evaluation.spark_session_utils import create_spark_session

teehr.__version__

## 1. Settings

In [ ]:
CONFIGURATION = "nwm30_medium_range"
OBSERVATIONS = "usgs_observations"
PREFIX = CONFIGURATION.split("_", 1)[0]  # nwm30: the configuration's location ids, nwm30-...

# The control plane API, from inside the cluster (KinD and use1 alike). From outside:
# https://control-plane.teehr.rtiamanzi.org (use1), https://control-plane.teehr.local.app.garden (KinD).
CONTROL_PLANE = os.environ.get(
    "CONTROL_PLANE_URL", "http://teehr-control-plane.teehr-hub.svc.cluster.local:8000"
)
VERIFY_TLS = "local.app.garden" not in CONTROL_PLANE  # KinD's certificate is self-signed

LEGACY = "iceberg.teehr"
PUBLIC = "iceberg.public"
GAUGES, OBSERVED, FORECAST, CROSSWALK = (
    "usgs_gauges", "usgs_streamflow", CONFIGURATION, f"usgs_{PREFIX}"
)

In [ ]:
# Local mode for counting and small copies; for a whole configuration start a cluster,
# e.g. create_spark_session(start_spark_cluster=True, executor_instances=32,
#                           executor_memory="16g", executor_cores=4)
spark = create_spark_session()
# The v1 tables declare required columns; the legacy columns are nullable in Spark's eyes.
spark.conf.set("spark.sql.iceberg.check-nullability", "false")

## 2. Count (writes nothing)

Rows per month of the forecasts (by `reference_time`, as they're copied) and of the
observations over the same `value_time` span; what units, variables and members they
use; the crosswalk and gauges.

In [ ]:
%%time
forecasts = spark.sql(f"""
    SELECT date_format(reference_time, 'yyyy-MM') AS month, count(*) AS rows,
           count(DISTINCT location_id) AS series
    FROM {LEGACY}.secondary_timeseries
    WHERE configuration_name = '{CONFIGURATION}'
    GROUP BY 1 ORDER BY 1
""").toPandas()
print(f"{forecasts.rows.sum():,} forecast rows in {len(forecasts)} months")
forecasts

In [ ]:
%%time
span = spark.sql(f"""
    SELECT min(value_time) AS first, max(value_time) AS last
    FROM {LEGACY}.secondary_timeseries WHERE configuration_name = '{CONFIGURATION}'
""").first()
observations = spark.sql(f"""
    SELECT date_format(value_time, 'yyyy-MM') AS month, count(*) AS rows
    FROM {LEGACY}.primary_timeseries
    WHERE configuration_name = '{OBSERVATIONS}'
      AND value_time >= '{span.first}' AND value_time <= '{span.last}'
      AND location_id IN (SELECT DISTINCT primary_location_id FROM {LEGACY}.location_crosswalks
                          WHERE secondary_location_id LIKE '{PREFIX}-%')
    GROUP BY 1 ORDER BY 1
""").toPandas()
print(f"forecasts value_time {span.first} .. {span.last}")
print(f"{observations.rows.sum():,} observation rows in {len(observations)} months")
observations

In [ ]:
for table, config in (("secondary_timeseries", CONFIGURATION), ("primary_timeseries", OBSERVATIONS)):
    spark.sql(f"""
        SELECT '{table}' AS table, unit_name, variable_name,
               {"member" if table == "secondary_timeseries" else "NULL"} AS member, count(*) AS rows
        FROM {LEGACY}.{table} WHERE configuration_name = '{config}'
        GROUP BY 2, 3, 4 ORDER BY 2, 3, 4
    """).show(truncate=False)
spark.sql(f"""
    SELECT count(*) AS crosswalk_rows, count(DISTINCT primary_location_id) AS gauges
    FROM {LEGACY}.location_crosswalks WHERE secondary_location_id LIKE '{PREFIX}-%'
""").show()

## 3. Register the v1 datasets in `public` (as you)

Datasets that already exist are left as they are. Values are streamflow in m3/s (the
legacy `m^3/s`): the count above must show only that unit.

In [ ]:
token = os.environ["POLARIS_USER_TOKEN"]  # your access token (JupyterHub sets it)
try:  # refresh it if it's about to expire (the dev kernel has the helper)
    from teehr.utils.keycloak_auth import refresh_user_token_from_env
    token = refresh_user_token_from_env()
except Exception:
    pass
api = requests.Session()
api.headers["Authorization"] = f"Bearer {token}"
api.verify = VERIFY_TLS


def register(body):
    url = f"{CONTROL_PLANE}/v1/projects/public/datasets"
    if api.get(f"{url}/{body['name']}").status_code == 200:
        print(f"public.{body['name']}: exists")
        return
    resp = api.post(url, json=body)
    resp.raise_for_status()
    print(f"public.{body['name']}: registered")


members = spark.sql(f"""
    SELECT count(*) AS n FROM {LEGACY}.secondary_timeseries
    WHERE configuration_name = '{CONFIGURATION}' AND member IS NOT NULL
""").first().n > 0
source = "Legacy TEEHR warehouse (copied)"
register({"type": "layer", "name": GAUGES, "source": source,
          "description": "USGS stream gauges"})
register({"type": "timeseries", "name": OBSERVED, "role": "observed",
          "variable": "streamflow", "statistic": "inst", "location_layer": GAUGES,
          "source": source, "description": "USGS instantaneous streamflow"})
register({"type": "timeseries", "name": FORECAST, "role": "forecast",
          "variable": "streamflow", "statistic": "inst",
          "optional_columns": ["member"] if members else [],
          "source": source, "description": f"{CONFIGURATION} (legacy copy)"})
register({"type": "crosswalk", "name": CROSSWALK, "left": OBSERVED, "right": FORECAST,
          "source": source, "description": f"USGS gauges to their {PREFIX} reaches"})

## 4. Copy

The layer, crosswalk and series tables first (small, replaced whole), then the
timeseries a month at a time.

In [ ]:
%%time
links = spark.sql(f"""
    SELECT DISTINCT primary_location_id AS left_key, secondary_location_id AS right_key
    FROM {LEGACY}.location_crosswalks WHERE secondary_location_id LIKE '{PREFIX}-%'
""").cache()
links.writeTo(f"{PUBLIC}.{CROSSWALK}").overwrite(F.lit(True))

gauges = spark.sql(f"""
    SELECT l.id AS location_key, l.name, l.geometry,
           ST_XMin(g) AS bbox_xmin, ST_YMin(g) AS bbox_ymin,
           ST_XMax(g) AS bbox_xmax, ST_YMax(g) AS bbox_ymax
    FROM (SELECT *, ST_GeomFromWKB(geometry) AS g FROM {LEGACY}.locations) l
    WHERE l.id IN (SELECT DISTINCT primary_location_id FROM {LEGACY}.location_crosswalks
                   WHERE secondary_location_id LIKE '{PREFIX}-%')
""")
gauges.writeTo(f"{PUBLIC}.{GAUGES}").overwrite(F.lit(True))

def series_rows(keys, located):
    return keys.select(
        F.col("series_key"),
        (F.col("series_key") if located else F.lit(None)).cast("string").alias("location_key"),
        *(F.lit(None).cast("string").alias(c) for c in ("location_layer", "period", "resolution")),
    )

series_rows(links.select(F.col("left_key").alias("series_key")).distinct(), True) \
    .writeTo(f"{PUBLIC}.{OBSERVED}_series").overwrite(F.lit(True))
series_rows(spark.sql(f"""
    SELECT DISTINCT location_id AS series_key FROM {LEGACY}.secondary_timeseries
    WHERE configuration_name = '{CONFIGURATION}'
"""), False).writeTo(f"{PUBLIC}.{FORECAST}_series").overwrite(F.lit(True))
print("layer, crosswalk and series tables copied")

In [ ]:
def copy_month(table, config, time_column, target, month, columns):
    """Replace one month (of ``time_column``) of ``target`` with the legacy rows."""
    rows = spark.sql(f"""
        SELECT {", ".join(columns)} FROM {LEGACY}.{table}
        WHERE configuration_name = '{config}'
          AND date_format({time_column}, 'yyyy-MM') = '{month}'
          {"AND location_id IN (SELECT left_key FROM links)" if table == "primary_timeseries" else ""}
          {f"AND value_time >= '{span.first}' AND value_time <= '{span.last}'" if table == "primary_timeseries" else ""}
    """)
    rows.writeTo(f"{PUBLIC}.{target}").overwritePartitions()


links.createOrReplaceTempView("links")
FORECAST_COLUMNS = ["location_id AS series_key", "reference_time", "value_time",
                    *(["member"] if members else []), "CAST(value AS DOUBLE) AS value"]
OBSERVED_COLUMNS = ["location_id AS series_key", "value_time", "CAST(value AS DOUBLE) AS value"]

START = None  # e.g. "2025-03" to resume from a month

In [ ]:
%%time
for month in forecasts.month:
    if START and month < START:
        continue
    copy_month("secondary_timeseries", CONFIGURATION, "reference_time", FORECAST, month,
               FORECAST_COLUMNS)
    print(f"{FORECAST} {month}: copied")

In [ ]:
%%time
for month in observations.month:
    if START and month < START:
        continue
    copy_month("primary_timeseries", OBSERVATIONS, "value_time", OBSERVED, month,
               OBSERVED_COLUMNS)
    print(f"{OBSERVED} {month}: copied")

## 5. Check: v1 rows per month against the legacy counts

In [ ]:
for target, legacy, column in ((FORECAST, forecasts, "reference_time"),
                               (OBSERVED, observations, "value_time")):
    v1 = spark.sql(f"""
        SELECT date_format({column}, 'yyyy-MM') AS month, count(*) AS v1_rows
        FROM {PUBLIC}.{target} GROUP BY 1
    """).toPandas()
    both = legacy[["month", "rows"]].merge(v1, on="month", how="outer").fillna(0)
    both["matches"] = both.rows == both.v1_rows
    print(f"{target}: {int(both.v1_rows.sum()):,} rows; every month matches: {both.matches.all()}")
    display(both[~both.matches])